<a href="https://colab.research.google.com/github/polreig/StartUp_DecoAI/blob/main/DecoAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Instalación

In [ ]:
!pip install -q diffusers transformers accelerate opencv-python Pillow gradio pydantic google-generativeai huggingface_hub

El Súper Motor de DecoAI

In [ ]:
import gradio as gr
import torch
import cv2
import json
import numpy as np
import urllib.parse
from PIL import Image, ImageDraw
from pydantic import BaseModel, Field
from typing import List, Dict, Tuple

from google import genai
from google.genai import types
from google.colab import userdata
from diffusers import StableDiffusionControlNetInpaintPipeline, ControlNetModel, DDIMScheduler
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation

print("INICIANDO DECO.AI...")

# 1. CREDENCIALES
try:
    GOOGLE_API_KEY = userdata.get('clave_API_gemini')
    gemini_client = genai.Client(api_key=GOOGLE_API_KEY)
except Exception as e:
    print("ADVERTENCIA: No se encontró 'clave_API_gemini' en los secretos de Colab.")

# 2. CARGA CENTRALIZADA DE MODELOS
print("Cargando IA de Segmentación (Mask2Former)...")
processor = AutoImageProcessor.from_pretrained("facebook/mask2former-swin-tiny-coco-panoptic")
segmentation_model = Mask2FormerForUniversalSegmentation.from_pretrained(
    "facebook/mask2former-swin-tiny-coco-panoptic", torch_dtype=torch.float16
).to("cuda")

print("Cargando IA Estructural (ControlNet MLSD)...")
controlnet = ControlNetModel.from_pretrained("lllyasviel/control_v11p_sd15_mlsd", torch_dtype=torch.float16)

print("Cargando IA Generativa (Inpainting)...")
pipe = StableDiffusionControlNetInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting", controlnet=controlnet, torch_dtype=torch.float16
).to("cuda")
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)


# ESQUEMAS DE DATOS
class ProductoRecomendado(BaseModel):
    nombre: str = Field(description="Nombre del producto")
    categoria: str = Field(description="'mueble', 'iluminacion', 'suelo', 'pared', 'electrodomestico', 'sanitario' o 'decoracion'")
    tienda_sugerida: str = Field(description="Leroy Merlin, IKEA, Amazon, Zara Home, Bauhaus, Kave Home")
    precio_estimado_eur: int
    detalles_medidas: str

class AnalisisDecoracionPro(BaseModel):
    tipo_estancia: str
    modo_generacion: str = Field(description="'completo' o 'parcial'")
    objeto_a_reemplazar: str = Field(description="Si es parcial, qué quitar. Si es completo: 'todo'")
    analisis_diseno: str
    prompt_generacion: str = Field(description="Prompt en inglés para Stable Diffusion")
    presupuesto_total_estimado: int
    lista_compra: List[ProductoRecomendado]


# 4. FUNCIONES CORE
def redimensionar(img, max_size=512):
    ancho, alto = img.size
    ratio = alto / ancho
    nuevo_ancho, nuevo_alto = (max_size, int(max_size * ratio)) if ancho > alto else (int(max_size / ratio), max_size)
    nuevo_ancho, nuevo_alto = (nuevo_ancho // 8) * 8, (nuevo_alto // 8) * 8
    return img.resize((nuevo_ancho, nuevo_alto), Image.Resampling.LANCZOS)

def extraer_mlsd(img):
    img_gray = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
    lsd = cv2.createLineSegmentDetector(0)
    lines, _, _, _ = lsd.detect(img_gray)
    drawn_img = np.zeros_like(img_gray)
    if lines is not None: lsd.drawSegments(drawn_img, lines)
    return Image.fromarray(drawn_img).convert("RGB")

def traducir_prompt(texto_es):
    """Usa Gemini como traductor invisible de Español a Inglés para Stable Diffusion"""
    if not texto_es.strip(): return ""
    prompt = f"Traduce este texto de diseño de interiores al inglés para usarlo en Stable Diffusion. Responde SOLO con la traducción directa, sin comillas: '{texto_es}'"
    response = gemini_client.models.generate_content(model='gemini-2.5-flash', contents=prompt)
    return response.text.strip()

def generar_mascara_auto(imagen, nombre_objeto):
    if nombre_objeto.lower() == 'todo': return Image.new("L", imagen.size, 255)
    
    inputs = processor(images=imagen, return_tensors="pt").to("cuda")
    if "pixel_values" in inputs: inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)
        
    with torch.no_grad(): outputs = segmentation_model(**inputs)
    
    segmentation_map = processor.post_process_panoptic_segmentation(outputs, target_sizes=[imagen.size[::-1]])[0]
    mapa = segmentation_map['segmentation'].cpu().numpy()
    info = segmentation_map['segments_info']
    mascara = np.zeros_like(mapa, dtype=np.uint8)
    
    target_ids = []
    obj = nombre_objeto.lower()
    if 'suelo' in obj: target_ids = [54, 55] 
    elif 'pared' in obj: target_ids = [51, 52] 
    elif 'cama' in obj: target_ids = [65]
    elif 'sofa' in obj or 'sillón' in obj: target_ids = [63]
    elif 'silla' in obj: target_ids = [62]
    elif 'mesa' in obj: target_ids = [67]
    elif 'inodoro' in obj or 'wc' in obj: target_ids = [70]
    elif 'lavabo' in obj: target_ids = [81]
    
    encontrado = False
    for segment in info:
        if segment['label_id'] in target_ids:
            mascara[mapa == segment['id']] = 255
            encontrado = True
            
    if not encontrado:
        w, h = imagen.size
        mascara_img = Image.new("L", (w, h), 0)
        draw = ImageDraw.Draw(mascara_img)
        draw.rectangle([w//4, h//4, 3*w//4, 3*h//4], fill=255)
        return mascara_img
    return Image.fromarray(mascara).convert("L")

def generar_html_tienda(datos):
    html = f"<div style='background:#fafafa; padding:20px; border-radius:10px;'><h3> Presupuesto: {datos['presupuesto_total_estimado']}€</h3><p><i>{datos['analisis_diseno']}</i></p><ul>"
    for item in datos['lista_compra']:
        query = urllib.parse.quote(item['nombre'])
        tienda = item['tienda_sugerida'].lower()
        link = f"https://www.google.com/search?q=comprar+{query}+{tienda}"
        if 'leroy' in tienda: link = f"https://www.leroymerlin.es/buscar?q={query}"
        elif 'ikea' in tienda: link = f"https://www.ikea.com/es/es/search/?q={query}"
        elif 'amazon' in tienda: link = f"https://www.amazon.es/s?k={query}"
        
        html += f"<li style='margin-bottom:10px; padding:10px; background:white; border-radius:5px;'><b>{item['nombre'].title()}</b> ({item['detalles_medidas']}) - ~{item['precio_estimado_eur']}€ <a href='{link}' target='_blank' style='margin-left:10px; background:#111; color:white; padding:4px 8px; border-radius:4px; text-decoration:none; font-size:12px;'>Ver en {item['tienda_sugerida']}</a></li>"
    html += "</ul></div>"
    return html


# 5. LÓGICA DE LAS PESTAÑAS 

def motor_automatico(img_entrada, peticion_es, medidas):
    if img_entrada is None: return None, None, "Sube una imagen primero."
    img_orig = redimensionar(Image.fromarray(img_entrada).convert("RGB"))
    
    prompt_gemini = f"Cliente pide en español: '{peticion_es}'. Medidas: '{medidas}'. Analiza y devuelve JSON. El prompt_generacion debe estar en INGLÉS."
    response = gemini_client.models.generate_content(
        model='gemini-2.5-flash',
        contents=[img_orig, prompt_gemini],
        config=types.GenerateContentConfig(response_mime_type="application/json", response_schema=AnalisisDecoracionPro)
    )
    datos = json.loads(response.text)
    
    mask_img = generar_mascara_auto(img_orig, datos['objeto_a_reemplazar'])
    fuerza = 0.95 if datos['modo_generacion'] == 'completo' else 0.80
    mlsd_img = extraer_mlsd(img_orig)
    
    img_final = pipe(
        prompt=datos['prompt_generacion'] + ", photorealistic, architectural digest, 8k",
        negative_prompt="cartoon, warped lines, messy, unrealistic lighting, deformed",
        image=img_orig, mask_image=mask_img, control_image=mlsd_img,
        num_inference_steps=30, controlnet_conditioning_scale=0.9, strength=fuerza, guidance_scale=8.0
    ).images[0]
    
    return img_final, mask_img, generar_html_tienda(datos)

def motor_manual(dict_imagen, peticion_es, fuerza):
    if dict_imagen is None or dict_imagen["background"] is None: return None
    
    img_orig = redimensionar(dict_imagen["background"].convert("RGB"))
    
    # LÓGICA DE MÁSCARA: Si pintas = Parcial. Si NO pintas = Total (Blanco).
    mascara_vacia = True
    if len(dict_imagen["layers"]) > 0:
        capa_alfa = dict_imagen["layers"][0].split()[-1]
        if capa_alfa.getextrema()[1] > 0: # Si detecta que hay al menos un pixel pintado
            img_mascara = capa_alfa.convert("L").resize(img_orig.size, Image.Resampling.LANCZOS)
            mascara_vacia = False
            print("Pincel detectado: Aplicando Cirugía Parcial.")
            
    if mascara_vacia:
        img_mascara = Image.new("L", img_orig.size, 255)
        print("Ningún trazo detectado: Aplicando Rediseño Total.")
    
    # LÓGICA DE IDIOMA: Traducción silenciosa
    print("🇪🇸->🇬🇧 Traduciendo prompt...")
    prompt_en = traducir_prompt(peticion_es)
    print(f"Prompt traducido: {prompt_en}")
    
    mlsd_img = extraer_mlsd(img_orig)
    
    img_final = pipe(
        prompt=prompt_en + ", photorealistic, interior design, 8k resolution",
        negative_prompt="cartoon, bad anatomy, warped lines, messy, outdoors",
        image=img_orig, mask_image=img_mascara, control_image=mlsd_img,
        num_inference_steps=30, controlnet_conditioning_scale=1.0, strength=fuerza, guidance_scale=8.5
    ).images[0]
    
    return img_final


# 6. LA INTERFAZ WEB DEFINITIVA (GRADIO) 
print("Levantando servidores web...")
with gr.Blocks(theme=gr.themes.Base(), title="DECO.AI") as app:
    gr.HTML("<center><h1>DECO.AI Studio</h1><p>Tu Asistente Integral de Diseño de Interiores</p></center>")
    
    with gr.Tabs():
        # PESTAÑA 1
        with gr.TabItem("Asistente Completo (Lista de Compra)"):
            with gr.Row():
                with gr.Column():
                    auto_in_img = gr.Image(label="Sube tu foto de la estancia")
                    auto_in_prompt = gr.Textbox(label="¿Qué quieres hacer?", placeholder="Ej: Cambia todo a estilo industrial con suelo de madera oscura")
                    auto_in_medidas = gr.Textbox(label="Medidas (Opcional)", placeholder="Ej: 15 metros cuadrados")
                    auto_btn = gr.Button("Analizar y Rediseñar", variant="primary")
                with gr.Column():
                    auto_out_img = gr.Image(label="Resultado Final")
                    auto_out_mask = gr.Image(label="Zona detectada por la IA")
            auto_out_html = gr.HTML(label="Lista de la Compra y Presupuesto")
            
            auto_btn.click(fn=motor_automatico, inputs=[auto_in_img, auto_in_prompt, auto_in_medidas], outputs=[auto_out_img, auto_out_mask, auto_out_html])

        # PESTAÑA 2
        with gr.TabItem("Diseñador Libre (Pincel)"):
            gr.Markdown("Pinta un mueble para cambiarlo, o **déjalo sin pintar para rediseñar toda la foto** respetando la estructura.")
            with gr.Row():
                with gr.Column():
                    manual_in_img = gr.ImageEditor(type="pil", label="Sube tu foto (Pinta el mueble o déjalo vacío)")
                    manual_in_prompt = gr.Textbox(label="¿Qué quieres ver aquí? (Escríbelo en Español)", placeholder="Ej: Una cama moderna de terciopelo verde oscuro")
                    manual_in_fuerza = gr.Slider(minimum=0.5, maximum=1.0, value=0.95, step=0.05, label="Fuerza del cambio (0.95 para cambios totales, 0.80 para muebles)")
                    manual_btn = gr.Button("Generar Diseño", variant="primary")
                with gr.Column():
                    manual_out_img = gr.Image(label="Nuevo Diseño")
            
            manual_btn.click(fn=motor_manual, inputs=[manual_in_img, manual_in_prompt, manual_in_fuerza], outputs=manual_out_img)

app.launch(share=True, debug=True)